# 04 · Inference & explainability

Uses the deployment checkpoint `ml/models/best_model.pth` through `ml.src.inference.WastePredictor` (Top-3, calibrated threshold, Grad-CAM) on held-out **test** images, and checks parity with the exported ONNX graph used by the API.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'ml' / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
DATA_DIR = os.environ.get('WASTEWISE_DATA_DIR', str(ROOT / 'ml/datasets/processed'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 160)

In [ ]:
from ml.src.inference import WastePredictor, overlay
from ml.src.preprocessing import load_rgb
pred = WastePredictor('ml/models/best_model.pth')
print('threshold', pred.threshold, 'temperature', pred.temperature)

In [ ]:
m = pd.read_csv('ml/datasets/manifests/manifest_v1.csv', dtype={'uid': str})
sample = pd.concat([m[(m.split == 'test') & (m.category == c)].sample(1, random_state=3) for c in sorted(m['category'].dropna().unique())]
                   + [m[m.split == 'ood'].sample(2, random_state=3)])
fig, axes = plt.subplots(2, len(sample), figsize=(2.3 * len(sample), 5.2))
for k, (_, row) in enumerate(sample.iterrows()):
    img = load_rgb(os.path.join(DATA_DIR, row['file'])); p = pred.predict(img)
    axes[0, k].imshow(img); axes[0, k].axis('off'); axes[0, k].set_title(f"true: {row['category'] if row['split'] != 'ood' else 'OOD'}", fontsize=8)
    axes[1, k].imshow(overlay(img, pred.gradcam(img, p.category))); axes[1, k].axis('off')
    lab = f'{p.category} {p.confidence:.2f}' if p.is_confident else f'uncertain ({p.category} {p.confidence:.2f})'
    axes[1, k].set_title(lab, fontsize=8, color='black' if p.is_confident else 'red')
plt.tight_layout()

## ONNX parity (the API runs `backend/model_artifacts/wastewise.onnx`)

In [ ]:
import onnxruntime as ort, torch
from ml.src.augmentation import eval_transforms
sess = ort.InferenceSession('backend/model_artifacts/wastewise.onnx', providers=['CPUExecutionProvider'])
diffs = []
for _, row in sample.iterrows():
    x = eval_transforms()(load_rgb(os.path.join(DATA_DIR, row['file']))).unsqueeze(0)
    with torch.no_grad(): ref, _ = pred.model(x)
    diffs.append(float(np.abs(sess.run(None, {'image': x.numpy()})[1] - ref.numpy()).max()))
print('max |logit difference| torch vs onnx:', max(diffs))